# Unified 128K Tokenizer Generation

This notebook creates a unified 128K tokenizer from three source tokenizers:
- **GPT-OSS** (gptoss_tokenizer.json) - Best Indic language support, used as base ordering
- **OLMo** (olmo_tokenizer.json) - Open dataset availability
- **Qwen** (qwen_tokenizer.json) - Strong general vocabulary

## Design Principles
1. **Position-based ordering**: Token position takes priority over model count
   - A token at position 300 (in 2 models) ranks higher than position 10000 (in 3 models)
2. **GPT-OSS as base**: Use GPT-OSS token order as the foundation
   - Tokens not in GPT-OSS are inserted based on their position in OLMo/Qwen
3. **Indic priority**: GPT-OSS has best Indic support; these tokens are preserved at their positions
4. **Code & JSON preserved**: Structural tokens and programming constructs maintained
5. **Merge consistency**: Only include merges where both tokens AND result exist in unified vocab

## Outputs
- `output/unified_128k_tokenizer.json` - Full HuggingFace-compatible tokenizer
- `output/unified_128k_vocab.json` - Detailed vocabulary with metadata
- `output/unified_128k_vocab_simple.json` - Simple token → id mapping

In [18]:
# Cell 1: Imports and Setup
import json
import os
import copy
import re
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

# Configuration
TARGET_VOCAB_SIZE = 128000
BASE_PATH = Path(".")
DATA_PATH = BASE_PATH / 'data'
OUTPUT_PATH = BASE_PATH / 'output'

# Source tokenizers (ordered by priority for Indic: GPT-OSS first)
SOURCE_TOKENIZERS = {
    'GPT-OSS': 'gptoss_tokenizer.json',  # Best Indic support
    'OLMo': 'olmo_tokenizer.json',        # Open dataset
    'Qwen': 'qwen_tokenizer.json',        # Strong general vocab
}

# Ensure output directory exists
OUTPUT_PATH.mkdir(exist_ok=True)

# GPT-2 style byte decoder for BPE tokens
def create_byte_decoder():
    """Create GPT-2 style byte decoder mapping."""
    bs = list(range(ord("!"), ord("~")+1)) + list(range(ord("¡"), ord("¬")+1)) + list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    return {chr(c): b for b, c in zip(bs, cs)}

def create_byte_encoder():
    """Create GPT-2 style byte encoder mapping (inverse of decoder)."""
    decoder = create_byte_decoder()
    return {v: k for k, v in decoder.items()}

BYTE_DECODER = create_byte_decoder()
BYTE_ENCODER = create_byte_encoder()

def decode_bpe_token(token):
    """Decode a BPE token to its actual string representation."""
    try:
        decoded_bytes = bytes([BYTE_DECODER.get(c, ord(c)) for c in token])
        return decoded_bytes.decode('utf-8', errors='replace')
    except:
        return token

def encode_to_bpe(text):
    """Encode text to BPE token representation."""
    try:
        return ''.join(BYTE_ENCODER.get(b, chr(b)) for b in text.encode('utf-8'))
    except:
        return text

print("=" * 70)
print("UNIFIED 128K TOKENIZER GENERATION")
print("=" * 70)
print(f"\nTarget vocabulary size: {TARGET_VOCAB_SIZE:,}")
print(f"Source tokenizers: {list(SOURCE_TOKENIZERS.keys())}")
print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")

UNIFIED 128K TOKENIZER GENERATION

Target vocabulary size: 128,000
Source tokenizers: ['GPT-OSS', 'OLMo', 'Qwen']
Data path: data
Output path: output


## 1. Load Source Tokenizers

Load all three tokenizers and extract vocabulary and merges.

In [19]:
# Cell 2: Load Source Tokenizers
tokenizer_data = {}
tokenizer_vocabs = {}
tokenizer_merges = {}
vocab_sizes = {}

print("Loading source tokenizers...")
print("-" * 50)

for name, filename in SOURCE_TOKENIZERS.items():
    filepath = DATA_PATH / filename
    
    if not filepath.exists():
        print(f"  ⚠️  {name}: File not found - {filename}")
        continue
    
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    tokenizer_data[name] = data
    vocab = data.get('model', {}).get('vocab', {})
    merges = data.get('model', {}).get('merges', [])
    
    tokenizer_vocabs[name] = vocab
    tokenizer_merges[name] = merges
    vocab_sizes[name] = len(vocab)
    
    print(f"  ✓ {name}: {len(vocab):,} tokens, {len(merges):,} merges")

print("-" * 50)
print(f"Loaded {len(tokenizer_vocabs)} tokenizers")

# Use GPT-OSS as the base structure (best Indic support)
BASE_TOKENIZER = 'GPT-OSS'
base_data = tokenizer_data[BASE_TOKENIZER]
print(f"\nBase tokenizer: {BASE_TOKENIZER}")

Loading source tokenizers...
--------------------------------------------------
  ✓ GPT-OSS: 199,998 tokens, 446,189 merges
  ✓ OLMo: 100,278 tokens, 100,000 merges
  ✓ Qwen: 151,643 tokens, 151,387 merges
--------------------------------------------------
Loaded 3 tokenizers

Base tokenizer: GPT-OSS


## 2. Token Classification Functions

Define functions for classifying tokens by language, script, and type (code/JSON).

In [20]:
# Cell 3: Token Classification Functions

# Unicode ranges for Indic scripts
INDIC_RANGES = {
    'Devanagari': (0x0900, 0x097F),
    'Bengali': (0x0980, 0x09FF),
    'Gurmukhi': (0x0A00, 0x0A7F),
    'Gujarati': (0x0A80, 0x0AFF),
    'Odia': (0x0B00, 0x0B7F),
    'Tamil': (0x0B80, 0x0BFF),
    'Telugu': (0x0C00, 0x0C7F),
    'Kannada': (0x0C80, 0x0CFF),
    'Malayalam': (0x0D00, 0x0D7F),
    'Sinhala': (0x0D80, 0x0DFF),
}

# CJK ranges (to identify Chinese tokens)
CJK_RANGES = [
    (0x4E00, 0x9FFF),   # CJK Unified Ideographs
    (0x3400, 0x4DBF),   # CJK Unified Ideographs Extension A
    (0x20000, 0x2A6DF), # CJK Unified Ideographs Extension B
    (0x2A700, 0x2B73F), # CJK Unified Ideographs Extension C
    (0x2B740, 0x2B81F), # CJK Unified Ideographs Extension D
]

def get_script_type(char):
    """Determine the script type of a character."""
    code = ord(char)
    
    # ASCII
    if 0x0000 <= code <= 0x007F:
        return 'ASCII'
    
    # Indic scripts
    for script, (start, end) in INDIC_RANGES.items():
        if start <= code <= end:
            return script
    
    # CJK
    for start, end in CJK_RANGES:
        if start <= code <= end:
            return 'CJK'
    
    return 'Other'

def classify_indic_script(text):
    """Classify text by Indic script. Returns script name or None."""
    script_counts = defaultdict(int)
    for char in text:
        code = ord(char)
        for script, (start, end) in INDIC_RANGES.items():
            if start <= code <= end:
                script_counts[script] += 1
                break
    
    if script_counts:
        return max(script_counts.items(), key=lambda x: x[1])[0]
    return None

def is_english_token(text):
    """Check if token is primarily English (Latin ASCII)."""
    if not text or len(text.strip()) == 0:
        return False
    clean = text.strip()
    ascii_chars = sum(1 for c in clean if ord(c) < 128 and (c.isalpha() or c.isdigit() or c in "'-_."))
    total = len(clean)
    return ascii_chars / max(total, 1) > 0.7

def is_code_token(token):
    """Check if token is related to programming/code."""
    decoded = decode_bpe_token(token).strip()
    
    code_patterns = [
        r'^(def|class|function|const|let|var|import|from|return|if|else|elif|for|while|try|except|catch|finally|async|await|yield|lambda|public|private|protected|static|void|int|float|double|string|bool|null|None|true|false|True|False|undefined|this|self|super|new|delete|throw|throws|extends|implements|interface|abstract|virtual|override|final|package|namespace|using|include|require|export|module)$',
        r'^(console|print|printf|println|log|debug|error|warn|info)$',
        r'^(Array|List|Dict|Map|Set|Object|String|Number|Boolean|Function|Promise|Observable)$',
        r'^\s*(//|#|/\*|\*/|<!--|-->)',
        r'^[{}\[\]();,.]$',
        r'^(=>|->|::|&&|\|\||==|!=|<=|>=|\+=|-=|\*=|/=|<<|>>)$',
        r'^__(init|main|name|str|repr|len|iter|next|call|getattr|setattr)__$',
        r'^(GET|POST|PUT|DELETE|PATCH|HTTP|API|REST|GraphQL)$',
        r'^(SELECT|INSERT|UPDATE|DELETE|FROM|WHERE|JOIN|ORDER|GROUP|BY|AND|OR|NOT|CREATE|DROP|TABLE|INDEX)$',
    ]
    
    for pattern in code_patterns:
        if re.search(pattern, decoded, re.IGNORECASE):
            return True
    
    # camelCase or snake_case
    if re.match(r'^[a-z]+[A-Z][a-zA-Z]*$', decoded):
        return True
    if re.match(r'^[a-z]+(_[a-z]+)+$', decoded):
        return True
    
    return False

def is_json_token(token):
    """Check if token is related to JSON handling."""
    decoded = decode_bpe_token(token).strip()
    
    json_patterns = [
        r'^[{}\[\]:,]$',
        r'^"[^"]*"$',
        r'^(true|false|null)$',
        r'^\s*(json|JSON|Json)$',
        r'^\s*(stringify|parse|dumps|loads|serialize|deserialize)$',
    ]
    
    for pattern in json_patterns:
        if re.search(pattern, decoded, re.IGNORECASE):
            return True
    
    return False

def normalize_token(token):
    """Normalize a token for cross-model comparison."""
    decoded = decode_bpe_token(token)
    normalized = decoded.strip()
    
    # Remove leading space markers
    if normalized.startswith('Ġ'):
        normalized = normalized[1:]
    if normalized.startswith('▁'):
        normalized = normalized[1:]
    if normalized.startswith(' '):
        normalized = normalized[1:]
    
    return normalized.lower() if normalized else ''

def classify_token(token):
    """
    Classify a token comprehensively.
    Returns: (category, is_indic, is_code, is_json, priority)
    Priority: 1=special, 2=code/json, 3=indic, 4=english, 5=other
    """
    decoded = decode_bpe_token(token).strip()
    
    if not decoded:
        return 'Empty', False, False, False, 99
    
    # Check code/JSON first (high priority)
    is_code = is_code_token(token)
    is_json = is_json_token(token)
    
    # Get script composition
    script_counts = defaultdict(int)
    for char in decoded:
        script = get_script_type(char)
        script_counts[script] += 1
    
    total_chars = len(decoded)
    ascii_pct = script_counts.get('ASCII', 0) / max(total_chars, 1)
    cjk_pct = script_counts.get('CJK', 0) / max(total_chars, 1)
    
    indic_count = sum(script_counts.get(s, 0) for s in INDIC_RANGES.keys())
    indic_pct = indic_count / max(total_chars, 1)
    
    # Classify
    if is_code or is_json:
        return 'Code/JSON', False, is_code, is_json, 2
    
    if indic_pct >= 0.5:
        indic_script = classify_indic_script(decoded)
        return indic_script or 'Indic', True, is_code, is_json, 3
    
    if cjk_pct >= 0.5:
        return 'CJK', False, is_code, is_json, 6
    
    if ascii_pct >= 0.7:
        return 'English', False, is_code, is_json, 4
    
    return 'Other', False, is_code, is_json, 5

print("Token classification functions defined.")
print(f"Indic scripts: {list(INDIC_RANGES.keys())}")

Token classification functions defined.
Indic scripts: ['Devanagari', 'Bengali', 'Gurmukhi', 'Gujarati', 'Odia', 'Tamil', 'Telugu', 'Kannada', 'Malayalam', 'Sinhala']


## 3. Build Cross-Tokenizer Token Registry

Create a registry of all tokens across the three tokenizers, tracking which models contain each token and their normalized positions.

In [33]:
# Cell 4: Build Token Registry

# CRITICAL FIX: Use raw_token as key to avoid collisions
# Previously: using normalized token caused '!' (pos 0) to be overwritten by '!čĊ' (pos 99074)
# because they both normalize to '!'

# Registry structure: raw_token -> {model: token_info}
token_registry = defaultdict(dict)

# Also track raw tokens for each model
raw_token_registry = defaultdict(dict)  # raw_token -> {model: token_id}

print("Building cross-tokenizer token registry...")
print("-" * 50)

for model_name in SOURCE_TOKENIZERS.keys():
    if model_name not in tokenizer_vocabs:
        continue
    
    vocab = tokenizer_vocabs[model_name]
    vocab_size = vocab_sizes[model_name]
    
    model_stats = {'total': 0, 'english': 0, 'indic': 0, 'code_json': 0, 'cjk': 0, 'other': 0}
    
    for raw_token, token_id in vocab.items():
        decoded = decode_bpe_token(raw_token)
        normalized = normalize_token(raw_token)
        
        # Skip empty tokens
        if not normalized and not decoded.strip():
            continue
        
        # Classify token
        category, is_indic, is_code, is_json, priority = classify_token(raw_token)
        
        # FIXED: Use raw_token as key to preserve all distinct tokens
        # This ensures single-byte tokens like '!' don't get overwritten by
        # multi-byte tokens like '!čĊ' that normalize to the same string
        key = raw_token
        
        token_registry[key][model_name] = {
            'raw_token': raw_token,
            'token_id': token_id,
            'decoded': decoded,
            'normalized': normalized,
            'category': category,
            'is_indic': is_indic,
            'is_code': is_code,
            'is_json': is_json,
            'priority': priority,
            'normalized_id': token_id / vocab_size
        }
        
        # Track raw tokens
        raw_token_registry[raw_token][model_name] = token_id
        
        # Update stats
        model_stats['total'] += 1
        if category == 'English':
            model_stats['english'] += 1
        elif is_indic:
            model_stats['indic'] += 1
        elif category == 'Code/JSON':
            model_stats['code_json'] += 1
        elif category == 'CJK':
            model_stats['cjk'] += 1
        else:
            model_stats['other'] += 1
    
    print(f"\n{model_name}:")
    print(f"  Total: {model_stats['total']:,}")
    print(f"  English: {model_stats['english']:,} | Indic: {model_stats['indic']:,}")
    print(f"  Code/JSON: {model_stats['code_json']:,} | CJK: {model_stats['cjk']:,} | Other: {model_stats['other']:,}")

print("-" * 50)
print(f"\nTotal unique raw tokens: {len(token_registry):,}")
print(f"Total unique raw tokens (cross-check): {len(raw_token_registry):,}")

Building cross-tokenizer token registry...
--------------------------------------------------

GPT-OSS:
  Total: 199,564
  English: 134,180 | Indic: 13,674
  Code/JSON: 1,167 | CJK: 7,522 | Other: 43,021

OLMo:
  Total: 99,837
  English: 92,235 | Indic: 43
  Code/JSON: 3,178 | CJK: 861 | Other: 3,520

Qwen:
  Total: 151,202
  English: 93,577 | Indic: 279
  Code/JSON: 3,180 | CJK: 25,561 | Other: 28,605
--------------------------------------------------

Total unique raw tokens: 252,311
Total unique raw tokens (cross-check): 252,311


## 4. Analyze Token Distribution

Analyze token overlap and distribution across the three source tokenizers.

In [34]:
# Cell 5: Analyze Token Distribution

# Build analysis rows
analysis_rows = []

for raw_token, model_data in token_registry.items():
    # Get consensus values
    categories = [d['category'] for d in model_data.values()]
    category = max(set(categories), key=categories.count)
    
    is_indic = any(d['is_indic'] for d in model_data.values())
    is_code = any(d['is_code'] for d in model_data.values())
    is_json = any(d['is_json'] for d in model_data.values())
    
    priorities = [d['priority'] for d in model_data.values()]
    priority = min(priorities)  # Use highest priority (lowest number)
    
    normalized_ids = [d['normalized_id'] for d in model_data.values()]
    avg_norm_id = np.mean(normalized_ids)
    
    # Get normalized form from model data
    normalized_token = None
    for pref_model in ['GPT-OSS', 'OLMo', 'Qwen']:
        if pref_model in model_data:
            normalized_token = model_data[pref_model].get('normalized', '')
            break
    
    analysis_rows.append({
        'normalized_token': normalized_token,
        'raw_token': raw_token,
        'model_count': len(model_data),
        'models': list(model_data.keys()),
        'category': category,
        'is_indic': is_indic,
        'is_code': is_code,
        'is_json': is_json,
        'priority': priority,
        'avg_norm_id': avg_norm_id,
        'model_data': model_data
    })

# Sort by: model_count (desc), priority (asc), avg_norm_id (asc)
analysis_rows.sort(key=lambda x: (-x['model_count'], x['priority'], x['avg_norm_id']))

print("=" * 70)
print("TOKEN DISTRIBUTION ANALYSIS")
print("=" * 70)

# Model count distribution
print("\nTokens by model count:")
for mc in [3, 2, 1]:
    subset = [r for r in analysis_rows if r['model_count'] == mc]
    indic_count = sum(1 for r in subset if r['is_indic'])
    code_json_count = sum(1 for r in subset if r['is_code'] or r['is_json'])
    print(f"  In {mc} model(s): {len(subset):>8,} (Indic: {indic_count:,}, Code/JSON: {code_json_count:,})")

# Category breakdown
print("\nCategory breakdown:")
category_counts = defaultdict(int)
for row in analysis_rows:
    category_counts[row['category']] += 1

for cat, count in sorted(category_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {cat:<20}: {count:>8,}")

# Indic tokens by source
print("\nIndic tokens by source tokenizer:")
for model in SOURCE_TOKENIZERS.keys():
    indic_count = sum(1 for r in analysis_rows if model in r['models'] and r['is_indic'])
    print(f"  {model}: {indic_count:,}")

TOKEN DISTRIBUTION ANALYSIS

Tokens by model count:
  In 3 model(s):   83,547 (Indic: 42, Code/JSON: 1,116)
  In 2 model(s):   31,198 (Indic: 207, Code/JSON: 2,063)
  In 1 model(s):  137,566 (Indic: 13,456, Code/JSON: 51)

Category breakdown:
  English             :  148,336
  Other               :   58,591
  CJK                 :   28,449
  Devanagari          :    3,993
  Code/JSON           :    3,230
  Bengali             :    2,136
  Malayalam           :    1,678
  Gujarati            :    1,621
  Telugu              :    1,337
  Kannada             :    1,312
  Tamil               :      978
  Gurmukhi            :      307
  Sinhala             :      300
  Odia                :       43

Indic tokens by source tokenizer:
  GPT-OSS: 13,674
  OLMo: 43
  Qwen: 279


In [35]:
# Cell 6: Build Unified Vocabulary - Position-Based Selection Strategy

print("=" * 70)
print("BUILDING UNIFIED VOCABULARY (Position-Based)")
print("=" * 70)

# NEW STRATEGY:
# Use GPT-OSS as base ordering, merge tokens from OLMo/Qwen coherently
# Position takes priority over model count:
#   - Token at position 300 (2 models) > Token at position 10000 (3 models)
# 
# Algorithm:
# 1. Start with GPT-OSS tokens in their original order
# 2. For tokens NOT in GPT-OSS, insert them at positions based on their
#    normalized position in OLMo/Qwen (maintaining order coherence)
# 3. Preserve Indic tokens from GPT-OSS (best Indic support)
# 4. Exclude CJK tokens to make room for more useful tokens

# Get GPT-OSS vocabulary with positions
gptoss_vocab = tokenizer_vocabs['GPT-OSS']
gptoss_size = vocab_sizes['GPT-OSS']

# Build candidate list with effective position for sorting
# Effective position = position in GPT-OSS (if present), else interpolated from OLMo/Qwen
candidates = []

for row in analysis_rows:
    normalized_token = row['normalized_token']
    raw_token = row['raw_token']
    model_data = row['model_data']
    
    # Skip CJK tokens
    if row['category'] == 'CJK':
        continue
    
    # Calculate effective position
    if 'GPT-OSS' in model_data:
        # Token exists in GPT-OSS - use its position directly
        gptoss_id = model_data['GPT-OSS']['token_id']
        effective_position = gptoss_id / gptoss_size  # Normalized 0-1
        source = 'GPT-OSS'
    else:
        # Token not in GPT-OSS - use position from OLMo or Qwen
        # Prefer the model with lower normalized position
        positions = []
        for model in ['OLMo', 'Qwen']:
            if model in model_data:
                norm_id = model_data[model]['normalized_id']
                positions.append((norm_id, model))
        
        if positions:
            # Use the lowest normalized position
            effective_position, source = min(positions, key=lambda x: x[0])
        else:
            # Fallback (shouldn't happen)
            effective_position = 1.0
            source = 'Unknown'
    
    # Create candidate entry
    candidates.append({
        'normalized_token': normalized_token,
        'raw_token': raw_token,
        'effective_position': effective_position,
        'source': source,
        'model_count': row['model_count'],
        'category': row['category'],
        'is_indic': row['is_indic'],
        'is_code': row['is_code'],
        'is_json': row['is_json'],
        'priority': row['priority'],
        'avg_norm_id': row['avg_norm_id'],
        'model_data': model_data,
        'models': row['models']
    })

print(f"\nTotal candidates (excluding CJK): {len(candidates):,}")

# Sort by effective position (primary) with Indic boost
# Indic tokens get a small position bonus to ensure they're included
def position_sort_key(c):
    pos = c['effective_position']
    # Indic tokens from GPT-OSS get slight priority at their position
    # (don't change position, but break ties in favor of Indic)
    indic_bonus = 0 if c['is_indic'] else 0.0001
    return (pos, indic_bonus, -c['model_count'])

candidates.sort(key=position_sort_key)

# Analysis of candidates by position ranges
print(f"\nPosition distribution:")
for threshold in [0.1, 0.25, 0.5, 0.75, 1.0]:
    count = sum(1 for c in candidates if c['effective_position'] <= threshold)
    indic_count = sum(1 for c in candidates if c['effective_position'] <= threshold and c['is_indic'])
    print(f"  Position ≤ {threshold:.2f}: {count:,} tokens (Indic: {indic_count:,})")

# Select top tokens up to target size
selected_tokens = candidates[:TARGET_VOCAB_SIZE]

print(f"\n{'='*50}")
print(f"SELECTED VOCABULARY SIZE: {len(selected_tokens):,}")
print(f"{'='*50}")

# Statistics on selected tokens
stats_selected = {
    'from_gptoss': sum(1 for t in selected_tokens if t['source'] == 'GPT-OSS'),
    'from_olmo': sum(1 for t in selected_tokens if t['source'] == 'OLMo'),
    'from_qwen': sum(1 for t in selected_tokens if t['source'] == 'Qwen'),
    'in_all_3': sum(1 for t in selected_tokens if t['model_count'] == 3),
    'in_2': sum(1 for t in selected_tokens if t['model_count'] == 2),
    'in_1': sum(1 for t in selected_tokens if t['model_count'] == 1),
    'indic': sum(1 for t in selected_tokens if t['is_indic']),
    'code': sum(1 for t in selected_tokens if t['is_code']),
    'json': sum(1 for t in selected_tokens if t['is_json']),
}

print(f"\nSource distribution:")
print(f"  From GPT-OSS order: {stats_selected['from_gptoss']:,}")
print(f"  From OLMo order: {stats_selected['from_olmo']:,}")
print(f"  From Qwen order: {stats_selected['from_qwen']:,}")

print(f"\nModel coverage:")
print(f"  In all 3 models: {stats_selected['in_all_3']:,}")
print(f"  In 2 models: {stats_selected['in_2']:,}")
print(f"  In 1 model: {stats_selected['in_1']:,}")

print(f"\nToken types:")
print(f"  Indic tokens: {stats_selected['indic']:,}")
print(f"  Code tokens: {stats_selected['code']:,}")
print(f"  JSON tokens: {stats_selected['json']:,}")

BUILDING UNIFIED VOCABULARY (Position-Based)

Total candidates (excluding CJK): 223,862

Position distribution:
  Position ≤ 0.10: 19,654 tokens (Indic: 1,069)
  Position ≤ 0.25: 49,834 tokens (Indic: 2,801)
  Position ≤ 0.50: 102,521 tokens (Indic: 6,310)
  Position ≤ 0.75: 159,409 tokens (Indic: 9,958)
  Position ≤ 1.00: 223,862 tokens (Indic: 13,705)

SELECTED VOCABULARY SIZE: 128,000

Source distribution:
  From GPT-OSS order: 116,376
  From OLMo order: 0
  From Qwen order: 11,624

Model coverage:
  In all 3 models: 63,932
  In 2 models: 21,065
  In 1 model: 43,003

Token types:
  Indic tokens: 7,792
  Code tokens: 2,710
  JSON tokens: 156


In [36]:
# Cell 7: Create Unified Vocabulary Mapping

# Create new vocab mapping: raw_token -> new_token_id
unified_vocab = {}
unified_metadata = []

for new_id, row in enumerate(selected_tokens):
    raw_token = row['raw_token']
    
    if raw_token is None:
        continue
    
    unified_vocab[raw_token] = new_id
    
    # Collect source token IDs
    source_ids = {}
    for model in SOURCE_TOKENIZERS.keys():
        if model in row['model_data']:
            source_ids[model] = row['model_data'][model]['token_id']
    
    metadata_entry = {
        'token': decode_bpe_token(raw_token),
        'raw_token': raw_token,
        'token_id': new_id,
        'normalized_form': row['normalized_token'],
        'category': row['category'],
        'is_indic': row['is_indic'],
        'is_code': row['is_code'],
        'is_json': row['is_json'],
        'model_count': row['model_count'],
        'source_token_ids': source_ids,
        'effective_position': round(row['effective_position'], 6),
        'position_source': row['source'],
        'avg_position': round(row['avg_norm_id'], 6)
    }
    unified_metadata.append(metadata_entry)

print(f"Unified vocabulary size: {len(unified_vocab):,}")

# Final statistics
stats = {
    'total': len(unified_metadata),
    'common_3': sum(1 for m in unified_metadata if m['model_count'] == 3),
    'indic': sum(1 for m in unified_metadata if m['is_indic']),
    'code': sum(1 for m in unified_metadata if m['is_code']),
    'json': sum(1 for m in unified_metadata if m['is_json']),
}

print(f"\nVocabulary composition:")
print(f"  Common to all 3 models: {stats['common_3']:,}")
print(f"  Indic tokens: {stats['indic']:,}")
print(f"  Code tokens: {stats['code']:,}")
print(f"  JSON tokens: {stats['json']:,}")

# Category breakdown
cat_counts = defaultdict(int)
for m in unified_metadata:
    cat_counts[m['category']] += 1

print(f"\nCategory breakdown:")
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1])[:10]:
    pct = count / len(unified_metadata) * 100
    print(f"  {cat:<20}: {count:>8,} ({pct:5.2f}%)")

Unified vocabulary size: 128,000

Vocabulary composition:
  Common to all 3 models: 63,932
  Indic tokens: 7,792
  Code tokens: 2,710
  JSON tokens: 156

Category breakdown:
  English             :   92,045 (71.91%)
  Other               :   25,378 (19.83%)
  Code/JSON           :    2,785 ( 2.18%)
  Devanagari          :    2,320 ( 1.81%)
  Bengali             :    1,161 ( 0.91%)
  Gujarati            :      947 ( 0.74%)
  Malayalam           :      944 ( 0.74%)
  Telugu              :      756 ( 0.59%)
  Kannada             :      718 ( 0.56%)
  Tamil               :      574 ( 0.45%)


## 5. Generate Consistent Merges

Filter merges to only include pairs where both tokens exist in the unified vocabulary, and their merged result also exists.

In [37]:
# Cell 8: Generate Consistent Merges

print("=" * 70)
print("GENERATING CONSISTENT MERGES")
print("=" * 70)

# Use GPT-OSS merges as base (best for Indic)
base_merges = tokenizer_merges[BASE_TOKENIZER]
print(f"\nBase tokenizer ({BASE_TOKENIZER}) merges: {len(base_merges):,}")

unified_vocab_set = set(unified_vocab.keys())

# Parse and filter merges
def parse_merge(merge):
    """Parse a merge entry into (token1, token2) tuple."""
    if isinstance(merge, list) and len(merge) == 2:
        return merge[0], merge[1]
    elif isinstance(merge, str):
        parts = merge.split(' ')
        if len(parts) == 2:
            return parts[0], parts[1]
    return None, None

# Filter merges: both tokens and merged result must be in vocab
consistent_merges = []
merge_stats = {'total': 0, 'valid': 0, 'missing_tokens': 0, 'missing_result': 0}

for merge in base_merges:
    merge_stats['total'] += 1
    
    token1, token2 = parse_merge(merge)
    if token1 is None:
        continue
    
    # Check if both tokens exist
    if token1 not in unified_vocab_set or token2 not in unified_vocab_set:
        merge_stats['missing_tokens'] += 1
        continue
    
    # Check if merged result exists
    merged_token = token1 + token2
    if merged_token not in unified_vocab_set:
        merge_stats['missing_result'] += 1
        continue
    
    # Valid merge
    consistent_merges.append(merge)
    merge_stats['valid'] += 1

print(f"\nMerge filtering results:")
print(f"  Total base merges: {merge_stats['total']:,}")
print(f"  Missing tokens: {merge_stats['missing_tokens']:,}")
print(f"  Missing result: {merge_stats['missing_result']:,}")
print(f"  Valid merges: {merge_stats['valid']:,}")

GENERATING CONSISTENT MERGES

Base tokenizer (GPT-OSS) merges: 446,189

Merge filtering results:
  Total base merges: 446,189
  Missing tokens: 75,938
  Missing result: 137,527
  Valid merges: 232,724


In [38]:
# Cell 9: Supplement Merges from Other Tokenizers

# Try to add valid merges from OLMo and Qwen that aren't duplicates
existing_merges = set()
for merge in consistent_merges:
    t1, t2 = parse_merge(merge)
    if t1 and t2:
        existing_merges.add((t1, t2))

additional_merges = []

for model_name in ['OLMo', 'Qwen']:
    if model_name not in tokenizer_merges:
        continue
    
    model_merges = tokenizer_merges[model_name]
    added = 0
    
    for merge in model_merges:
        token1, token2 = parse_merge(merge)
        if token1 is None:
            continue
        
        # Skip if already have this merge
        if (token1, token2) in existing_merges:
            continue
        
        # Check if all tokens exist in unified vocab
        if token1 not in unified_vocab_set or token2 not in unified_vocab_set:
            continue
        
        merged_token = token1 + token2
        if merged_token not in unified_vocab_set:
            continue
        
        # Add merge
        additional_merges.append(merge)
        existing_merges.add((token1, token2))
        added += 1
    
    print(f"Additional valid merges from {model_name}: {added:,}")

# Combine all merges
all_merges = consistent_merges + additional_merges

print(f"\nTotal consistent merges: {len(all_merges):,}")

Additional valid merges from OLMo: 12,659
Additional valid merges from Qwen: 86

Total consistent merges: 245,469


In [39]:
# Cell 10: Build Complete Tokenizer Structure

print("=" * 70)
print("BUILDING TOKENIZER STRUCTURE")
print("=" * 70)

# Start with base tokenizer structure
unified_tokenizer = copy.deepcopy(base_data)

# Update vocabulary
unified_tokenizer['model']['vocab'] = unified_vocab

# Update merges
unified_tokenizer['model']['merges'] = all_merges

# Update added_tokens (filter to only include tokens in unified vocab)
if 'added_tokens' in unified_tokenizer:
    original_added = unified_tokenizer['added_tokens']
    filtered_added = []
    
    for token_entry in original_added:
        content = token_entry.get('content', '')
        if content in unified_vocab:
            new_entry = copy.deepcopy(token_entry)
            new_entry['id'] = unified_vocab[content]
            filtered_added.append(new_entry)
    
    unified_tokenizer['added_tokens'] = filtered_added
    print(f"Added tokens: {len(original_added)} -> {len(filtered_added)}")

# Verify structure
print(f"\nUnified tokenizer structure:")
print(f"  Vocabulary size: {len(unified_tokenizer['model']['vocab']):,}")
print(f"  Merges: {len(unified_tokenizer['model']['merges']):,}")
if 'added_tokens' in unified_tokenizer:
    print(f"  Added tokens: {len(unified_tokenizer['added_tokens']):,}")

BUILDING TOKENIZER STRUCTURE
Added tokens: 21 -> 0

Unified tokenizer structure:
  Vocabulary size: 128,000
  Merges: 245,469
  Added tokens: 0


## 6. Save Output Files

Save the unified tokenizer and vocabulary files.

In [40]:
# Cell 11: Save Unified Tokenizer (HuggingFace Format)

tokenizer_output_path = OUTPUT_PATH / 'unified_128k_tokenizer.json'

with open(tokenizer_output_path, 'w', encoding='utf-8') as f:
    json.dump(unified_tokenizer, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(tokenizer_output_path) / (1024 * 1024)
print(f"✓ Saved unified tokenizer: {tokenizer_output_path}")
print(f"  File size: {file_size:.2f} MB")

✓ Saved unified tokenizer: output\unified_128k_tokenizer.json
  File size: 15.61 MB


In [41]:
# Cell 12: Save Vocabulary with Metadata

vocab_output_path = OUTPUT_PATH / 'unified_128k_vocab.json'

vocab_json = {
    "name": "Unified 128K Vocabulary",
    "description": "Unified vocabulary from GPT-OSS, OLMo, and Qwen tokenizers",
    "version": "1.0",
    "vocab_size": len(unified_vocab),
    "source_tokenizers": list(SOURCE_TOKENIZERS.keys()),
    "base_tokenizer": BASE_TOKENIZER,
    "statistics": {
        "total_tokens": len(unified_metadata),
        "common_to_all": stats['common_3'],
        "indic_tokens": stats['indic'],
        "code_tokens": stats['code'],
        "json_tokens": stats['json'],
    },
    "category_breakdown": dict(cat_counts),
    "tokens": unified_metadata
}

with open(vocab_output_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_json, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(vocab_output_path) / (1024 * 1024)
print(f"✓ Saved vocabulary with metadata: {vocab_output_path}")
print(f"  File size: {file_size:.2f} MB")

✓ Saved vocabulary with metadata: output\unified_128k_vocab.json
  File size: 57.10 MB


In [42]:
# Cell 13: Save Simple Vocabulary Mapping

simple_vocab_path = OUTPUT_PATH / 'unified_128k_vocab_simple.json'

# Simple mapping: decoded_token -> id
simple_mapping = {}
for entry in unified_metadata:
    simple_mapping[entry['token']] = entry['token_id']

with open(simple_vocab_path, 'w', encoding='utf-8') as f:
    json.dump(simple_mapping, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(simple_vocab_path) / (1024 * 1024)
print(f"✓ Saved simple vocab mapping: {simple_vocab_path}")
print(f"  File size: {file_size:.2f} MB")

✓ Saved simple vocab mapping: output\unified_128k_vocab_simple.json
  File size: 2.52 MB


## 7. Verification and Summary

Verify the generated tokenizer and display final summary.

In [43]:
# Cell 14: Verify Generated Tokenizer

print("=" * 70)
print("VERIFICATION")
print("=" * 70)

# Reload and verify
with open(tokenizer_output_path, 'r', encoding='utf-8') as f:
    verify_data = json.load(f)

verify_vocab = verify_data.get('model', {}).get('vocab', {})
verify_merges = verify_data.get('model', {}).get('merges', [])

print(f"\nTokenizer verification:")
print(f"  Vocabulary size: {len(verify_vocab):,}")
print(f"  Merges: {len(verify_merges):,}")

# Verify merge consistency
print(f"\nMerge consistency check:")
verify_vocab_set = set(verify_vocab.keys())
invalid_merges = 0

for merge in verify_merges[:1000]:  # Check first 1000
    t1, t2 = parse_merge(merge)
    if t1 and t2:
        if t1 not in verify_vocab_set or t2 not in verify_vocab_set:
            invalid_merges += 1
        merged = t1 + t2
        if merged not in verify_vocab_set:
            invalid_merges += 1

if invalid_merges == 0:
    print(f"  ✓ All sampled merges are consistent")
else:
    print(f"  ⚠ Found {invalid_merges} inconsistent merges")

# Sample tokens
print(f"\nSample tokens from unified vocabulary:")
sample_indices = [0, 1, 2, 100, 1000, 10000, 50000, 100000, len(unified_metadata)-1]
for idx in sample_indices:
    if idx < len(unified_metadata):
        entry = unified_metadata[idx]
        print(f"  [{idx:>6}] '{entry['token'][:30]:<30}' | {entry['category']:<15} | models: {entry['model_count']}")

VERIFICATION

Tokenizer verification:
  Vocabulary size: 128,000
  Merges: 245,469

Merge consistency check:
  ✓ All sampled merges are consistent

Sample tokens from unified vocabulary:
  [     0] '!                             ' | English         | models: 3
  [     1] '"                             ' | English         | models: 3
  [     2] '#                             ' | Code/JSON       | models: 3
  [   100] '�                             ' | Other           | models: 3
  [  1000] 'ش                             ' | Other           | models: 3
  [ 10000] 'shot                          ' | English         | models: 3
  [ 50000] ' enterprises                  ' | English         | models: 3
  [100000] ' ആരംഭ                         ' | Malayalam       | models: 1
  [127999] ' Ј                            ' | Other           | models: 1


In [44]:
# Cell 15: Final Summary

print("=" * 70)
print("UNIFIED 128K TOKENIZER GENERATION - COMPLETE")
print("=" * 70)

print(f"\nSource Tokenizers:")
for model, filename in SOURCE_TOKENIZERS.items():
    size = vocab_sizes.get(model, 0)
    print(f"  • {model}: {size:,} tokens ({filename})")

print(f"\nUnified Tokenizer:")
print(f"  • Vocabulary size: {len(unified_vocab):,}")
print(f"  • Consistent merges: {len(all_merges):,}")
print(f"  • Base structure: {BASE_TOKENIZER}")

print(f"\nToken Composition:")
print(f"  • Common to all 3 models: {stats['common_3']:,}")
print(f"  • Indic tokens: {stats['indic']:,}")
print(f"  • Code tokens: {stats['code']:,}")
print(f"  • JSON tokens: {stats['json']:,}")

print(f"\nOutput Files:")
print(f"  1. {tokenizer_output_path}")
print(f"     → Full HuggingFace-compatible tokenizer")
print(f"  2. {vocab_output_path}")
print(f"     → Vocabulary with detailed metadata")
print(f"  3. {simple_vocab_path}")
print(f"     → Simple token → id mapping")

# File sizes
print(f"\nFile Sizes:")
for path in [tokenizer_output_path, vocab_output_path, simple_vocab_path]:
    if path.exists():
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  • {path.name}: {size_mb:.2f} MB")

print(f"\n{'='*70}")
print("✓ Tokenizer generation complete!")
print("="*70)

UNIFIED 128K TOKENIZER GENERATION - COMPLETE

Source Tokenizers:
  • GPT-OSS: 199,998 tokens (gptoss_tokenizer.json)
  • OLMo: 100,278 tokens (olmo_tokenizer.json)
  • Qwen: 151,643 tokens (qwen_tokenizer.json)

Unified Tokenizer:
  • Vocabulary size: 128,000
  • Consistent merges: 245,469
  • Base structure: GPT-OSS

Token Composition:
  • Common to all 3 models: 63,932
  • Indic tokens: 7,792
  • Code tokens: 2,710
  • JSON tokens: 156

Output Files:
  1. output\unified_128k_tokenizer.json
     → Full HuggingFace-compatible tokenizer
  2. output\unified_128k_vocab.json
     → Vocabulary with detailed metadata
  3. output\unified_128k_vocab_simple.json
     → Simple token → id mapping

File Sizes:
  • unified_128k_tokenizer.json: 15.61 MB
  • unified_128k_vocab.json: 57.10 MB
  • unified_128k_vocab_simple.json: 2.52 MB

✓ Tokenizer generation complete!
